In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np

In [ ]:
# 1. train 데이터에 최종 결정된 파생변수 추가하기

In [22]:
df = pd.read_csv('./train_heat.csv', index_col=0)
df.columns = ['tm', 'branch_id', 'ta', 'wd', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'heat_demand']
df.head(24)

,tm,branch_id,ta,wd,ws,rn_day,rn_hr1,hm,si,ta_chi,heat_demand
1,2021010101,A,-10.1,78.3,0.5,0.0,0.0,68.2,-99.00,-8.2,281
2,2021010102,A,-10.2,71.9,0.6,0.0,0.0,69.9,-99.00,-8.6,262
3,2021010103,A,-10.0,360.0,0.0,0.0,0.0,69.2,-99.00,-8.8,266
4,2021010104,A,-9.3,155.9,0.5,0.0,0.0,65.0,-99.00,-8.9,285
5,2021010105,A,-9.0,74.3,1.9,0.0,0.0,63.5,-99.00,-9.2,283
6,2021010106,A,-9.0,81.9,1.5,0.0,0.0,66.9,-99.00,-9.2,285
7,2021010107,A,-9.0,88.4,1.3,0.0,0.0,64.7,-99.00,-8.4,285
8,2021010108,A,-8.6,88.9,1.2,0.0,0.0,63.8,0.00,-8.4,284
9,2021010109,A,-7.4,71.6,1.3,0.0,0.0,61.2,0.27,-7.5,290
10,2021010110,A,-4.7,157.5,0.5,0.0,0.0,50.4,0.76,-5.3,285


In [24]:
# 선택된 열 범위에서 -98 이하인 데이터 null 변환
import numpy as np
cols_to_convert = ['ta_chi', 'heat_demand']
df[cols_to_convert] = df[cols_to_convert].apply(pd.to_numeric, errors='coerce').astype(float)
df[cols_to_convert] = df[cols_to_convert].mask(df[cols_to_convert] <= -98, np.nan)

In [26]:
df.isna().sum()

tm              0
branch_id       0
ta              0
wd              0
ws              0
rn_day          0
rn_hr1          0
hm              0
si              0
ta_chi         20
heat_demand    23
dtype: int64

In [30]:
# 선택된 열 범위에서 -9.9 인 데이터 null 변환
df['wd'] = df['wd'].apply(pd.to_numeric, errors='coerce').astype(float)
df['wd'] = df['wd'].mask(df['wd'] == -9.9, np.nan)

In [32]:
df.isna().sum()

tm                0
branch_id         0
ta                0
wd             1589
ws                0
rn_day            0
rn_hr1            0
hm                0
si                0
ta_chi           20
heat_demand      23
dtype: int64

In [5]:
def create_common_features(df):
    df_copy = df.copy() # 원본 DF 손상 방지
    df_copy['datetime'] = pd.to_datetime(df_copy['datetime'], errors='coerce')
    df_copy['year'] = df_copy['datetime'].dt.year
    df_copy['month'] = df_copy['datetime'].dt.month
    df_copy['day'] = df_copy['datetime'].dt.day
    df_copy['hour'] = df_copy['datetime'].dt.hour
    df_copy['dayofweek'] = df_copy['datetime'].dt.dayofweek
    df_copy['dayofyear'] = df_copy['datetime'].dt.dayofyear
    df_copy['weekofyear'] = df_copy['datetime'].dt.isocalendar().week.astype(int)

    
    # branch_id 인코딩
    le = LabelEncoder()
    df_copy['branch_id_encoded'] = le.fit_transform(df_copy['branch_id'])
    
    return df_copy

In [111]:
df4 = df[['branch_id', 'ta_chi', 'heat_demand','datetime']]
df4.head()

,ta_chi,branch_id,heat_demand,datetime
0,-8.2,A,281.0,2021-01-01 01:00:00
1,-8.6,A,262.0,2021-01-01 02:00:00
2,-8.8,A,266.0,2021-01-01 03:00:00
3,-8.9,A,285.0,2021-01-01 04:00:00
4,-9.2,A,283.0,2021-01-01 05:00:00


In [113]:
# 모델 4
df4 = pd.read_csv('./train_heat_base.csv', encoding='utf-8')
df4 = create_common_features(df4)
df4['sin_hour'] = np.sin(2 * np.pi * df4['hour'] / 24.0)
df4['cos_hour'] = np.cos(2 * np.pi * df4['hour'] / 24.0)
df4['julius_time'] = df4['datetime'].apply(lambda x: x.toordinal())
df4['ta_chi_lag3'] = df4['ta_chi'].shift(3)
df4['ta_chi_lag24'] = df4['ta_chi'].shift(24)
df4['ta_chi_roll_mean_3'] = df4['ta_chi'].shift(1).rolling(window=3).mean()
df4['ta_chi_roll_std_3'] = df4['ta_chi'].shift(1).rolling(window=3).std()
df4 = df4.dropna()
df4.head()

,ta_chi,branch_id,heat_demand,datetime,year,month,day,hour,dayofweek,dayofyear,weekofyear,branch_id_encoded,sin_hour,cos_hour,julius_time,ta_chi_lag3,ta_chi_lag24,ta_chi_roll_mean_3,ta_chi_roll_std_3
24,-6.9,A,279.0,2021-01-02 01:00:00,2021,1,2,1,5,2,53,0,0.258819,0.965926,737792,-4.9,-8.2,-5.466667,0.981495
25,-7.9,A,272.0,2021-01-02 02:00:00,2021,1,2,2,5,2,53,0,0.500000,0.866025,737792,-4.9,-8.6,-6.133333,1.078579
26,-5.5,A,264.0,2021-01-02 03:00:00,2021,1,2,3,5,2,53,0,0.707107,0.707107,737792,-6.6,-8.8,-7.133333,0.680686
27,-6.0,A,269.0,2021-01-02 04:00:00,2021,1,2,4,5,2,53,0,0.866025,0.500000,737792,-6.9,-8.9,-6.766667,1.205543
28,-6.4,A,284.0,2021-01-02 05:00:00,2021,1,2,5,5,2,53,0,0.965926,0.258819,737792,-7.9,-9.2,-6.466667,1.266228


In [115]:
import holidays
# 출퇴근(7-9시, 19-22시) 유무: 1, 0
df4['work_time'] = 0
target_data = [7, 8, 9, 19, 20, 21, 22]
df4.loc[df4['hour'].isin(target_data), 'work_time'] = 1
# 주말 유무(1, 0)
df4['weekend'] = 0
target_data = [5, 6]
df4.loc[df4['dayofweek'].isin(target_data), 'weekend'] = 1
# 공휴일 유무(1, 0)
kr_holidays = holidays.KR(years=range(2023, 2025))
df4['date_only'] = df4['datetime'].dt.date
holiday_dates_set = set(kr_holidays)
df4['holiday'] = df4['date_only'].isin(holiday_dates_set).astype(int)
df4.drop(columns=['date_only'], inplace=True)

In [117]:
df4.isna().sum()

ta_chi                0
branch_id             0
heat_demand           0
datetime              0
year                  0
month                 0
day                   0
hour                  0
dayofweek             0
dayofyear             0
weekofyear            0
branch_id_encoded     0
sin_hour              0
cos_hour              0
julius_time           0
ta_chi_lag3           0
ta_chi_lag24          0
ta_chi_roll_mean_3    0
ta_chi_roll_std_3     0
work_time             0
weekend               0
holiday               0
dtype: int64

In [119]:
df4.to_csv('./train_heat_add_time_features.csv', index=False, encoding='utf-8')

In [ ]:
# 2-1. 결과 제출용 Test 데이터프레임 만들기

In [67]:
df = pd.read_csv('./test_heat.csv', encoding='utf-8')
df.head()

,TM,branch_ID,TA,WD,WS,RN_DAY,RN_HR1,HM,SI,ta_chi,heat_demand
0,2024010100,A,0.5,171.3,0.8,2.5,0.0,97.1,-99.0,0.3,NaN
1,2024010101,A,0.4,93.7,1.0,0.0,0.0,96.8,-99.0,0.1,NaN
2,2024010102,A,-0.1,133.0,0.8,0.0,0.0,97.0,-99.0,0.0,NaN
3,2024010103,A,-0.8,218.6,0.6,0.0,0.0,96.9,-99.0,-0.2,NaN
4,2024010104,A,0.1,58.7,1.5,0.0,0.0,97.0,-99.0,-0.1,NaN


In [69]:
result_df = df.drop('heat_demand',axis=1)
result_df.head()
result_df.to_csv('./result_base.csv', index=False, encoding='utf-8')

,TM,branch_ID,TA,WD,WS,RN_DAY,RN_HR1,HM,SI,ta_chi
0,2024010100,A,0.5,171.3,0.8,2.5,0.0,97.1,-99.0,0.3
1,2024010101,A,0.4,93.7,1.0,0.0,0.0,96.8,-99.0,0.1
2,2024010102,A,-0.1,133.0,0.8,0.0,0.0,97.0,-99.0,0.0
3,2024010103,A,-0.8,218.6,0.6,0.0,0.0,96.9,-99.0,-0.2
4,2024010104,A,0.1,58.7,1.5,0.0,0.0,97.0,-99.0,-0.1


In [ ]:
# 2-2. 학습용 Test 데이터프레임 만들기

In [71]:
df = pd.read_csv('./test_heat.csv', encoding='utf-8')
df.head()

,TM,branch_ID,TA,WD,WS,RN_DAY,RN_HR1,HM,SI,ta_chi,heat_demand
0,2024010100,A,0.5,171.3,0.8,2.5,0.0,97.1,-99.0,0.3,NaN
1,2024010101,A,0.4,93.7,1.0,0.0,0.0,96.8,-99.0,0.1,NaN
2,2024010102,A,-0.1,133.0,0.8,0.0,0.0,97.0,-99.0,0.0,NaN
3,2024010103,A,-0.8,218.6,0.6,0.0,0.0,96.9,-99.0,-0.2,NaN
4,2024010104,A,0.1,58.7,1.5,0.0,0.0,97.0,-99.0,-0.1,NaN


In [73]:
df.columns = ['tm', 'branch_id', 'ta', 'wd', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'heat_demand']
df.head()

,tm,branch_id,ta,wd,ws,rn_day,rn_hr1,hm,si,ta_chi,heat_demand
0,2024010100,A,0.5,171.3,0.8,2.5,0.0,97.1,-99.0,0.3,NaN
1,2024010101,A,0.4,93.7,1.0,0.0,0.0,96.8,-99.0,0.1,NaN
2,2024010102,A,-0.1,133.0,0.8,0.0,0.0,97.0,-99.0,0.0,NaN
3,2024010103,A,-0.8,218.6,0.6,0.0,0.0,96.9,-99.0,-0.2,NaN
4,2024010104,A,0.1,58.7,1.5,0.0,0.0,97.0,-99.0,-0.1,NaN


In [75]:
df = df.drop('heat_demand',axis=1)

In [81]:
import numpy as np
df['ta_chi'] = df['ta_chi'].apply(pd.to_numeric, errors='coerce').astype(float)
df['ta_chi'] = df['ta_chi'].mask(df['ta_chi'] <= -98, np.nan)

In [83]:
df.isna().sum()

tm           0
branch_id    0
ta           0
wd           0
ws           0
rn_day       0
rn_hr1       0
hm           0
si           0
ta_chi       1
dtype: int64

In [85]:
df['wd'] = df['wd'].apply(pd.to_numeric, errors='coerce').astype(float)
df['wd'] = df['wd'].mask(df['wd'] == -9.9, np.nan)

In [87]:
df.isna().sum()

tm            0
branch_id     0
ta            0
wd           25
ws            0
rn_day        0
rn_hr1        0
hm            0
si            0
ta_chi        1
dtype: int64

In [89]:
df=df.dropna(subset=['wd'])
df.isna().sum()

tm           0
branch_id    0
ta           0
wd           0
ws           0
rn_day       0
rn_hr1       0
hm           0
si           0
ta_chi       1
dtype: int64

In [45]:
df

,tm,branch_id,ta,wd,ws,rn_day,rn_hr1,hm,si,ta_chi
0,2024010100,A,0.5,171.3,0.8,2.5,0.0,97.1,-99.0,0.3
1,2024010101,A,0.4,93.7,1.0,0.0,0.0,96.8,-99.0,0.1
2,2024010102,A,-0.1,133.0,0.8,0.0,0.0,97.0,-99.0,0.0
3,2024010103,A,-0.8,218.6,0.6,0.0,0.0,96.9,-99.0,-0.2
4,2024010104,A,0.1,58.7,1.5,0.0,0.0,97.0,-99.0,-0.1
...,...,...,...,...,...,...,...,...,...,...
166910,2024123120,S,-1.1,360.0,0.0,0.0,0.0,45.8,-99.0,-1.7
166911,2024123121,S,-1.3,360.0,0.0,0.0,0.0,48.3,-99.0,-2.3
166912,2024123122,S,-2.4,360.0,0.0,0.0,0.0,60.0,-99.0,-3.1
166913,2024123123,S,-3.6,360.0,0.0,0.0,0.0,65.7,-99.0,-3.9


In [91]:
# 선형보간
df = df.interpolate(method='linear').ffill().bfill()
df.isna().sum()

C:\Users\jioyu\AppData\Local\Temp\ipykernel_26856\2401544329.py:2: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.interpolate(method='linear').ffill().bfill()


tm           0
branch_id    0
ta           0
wd           0
ws           0
rn_day       0
rn_hr1       0
hm           0
si           0
ta_chi       0
dtype: int64

In [93]:
df['datetime'] = pd.to_datetime(df['tm'], format='%Y%m%d%H', errors='coerce')
df = df.drop('tm', axis=1)
df.head()

,branch_id,ta,wd,ws,rn_day,rn_hr1,hm,si,ta_chi,datetime
0,A,0.5,171.3,0.8,2.5,0.0,97.1,-99.0,0.3,2024-01-01 00:00:00
1,A,0.4,93.7,1.0,0.0,0.0,96.8,-99.0,0.1,2024-01-01 01:00:00
2,A,-0.1,133.0,0.8,0.0,0.0,97.0,-99.0,0.0,2024-01-01 02:00:00
3,A,-0.8,218.6,0.6,0.0,0.0,96.9,-99.0,-0.2,2024-01-01 03:00:00
4,A,0.1,58.7,1.5,0.0,0.0,97.0,-99.0,-0.1,2024-01-01 04:00:00


In [95]:
df = df[['branch_id', 'ta_chi', 'datetime']]
df.head()

,branch_id,ta_chi,datetime
0,A,0.3,2024-01-01 00:00:00
1,A,0.1,2024-01-01 01:00:00
2,A,0.0,2024-01-01 02:00:00
3,A,-0.2,2024-01-01 03:00:00
4,A,-0.1,2024-01-01 04:00:00


In [97]:
df4 = df.copy()
df4 = create_common_features(df4)
df4['sin_hour'] = np.sin(2 * np.pi * df4['hour'] / 24.0)
df4['cos_hour'] = np.cos(2 * np.pi * df4['hour'] / 24.0)
df4['julius_time'] = df4['datetime'].apply(lambda x: x.toordinal())
df4['ta_chi_lag3'] = df4['ta_chi'].shift(3)
df4['ta_chi_lag24'] = df4['ta_chi'].shift(24)
df4['ta_chi_roll_mean_3'] = df4['ta_chi'].shift(1).rolling(window=3).mean()
df4['ta_chi_roll_std_3'] = df4['ta_chi'].shift(1).rolling(window=3).std()

branch_id    0
ta_chi       0
datetime     0
dtype: int64

In [99]:
df4.isna().sum()

branch_id              0
ta_chi                 0
datetime               0
year                   0
month                  0
day                    0
hour                   0
dayofweek              0
dayofyear              0
weekofyear             0
branch_id_encoded      0
sin_hour               0
cos_hour               0
julius_time            0
ta_chi_lag3            3
ta_chi_lag24          24
ta_chi_roll_mean_3     3
ta_chi_roll_std_3      3
dtype: int64

In [103]:
df4 =df4.dropna()
df4.isna().sum()

branch_id             0
ta_chi                0
datetime              0
year                  0
month                 0
day                   0
hour                  0
dayofweek             0
dayofyear             0
weekofyear            0
branch_id_encoded     0
sin_hour              0
cos_hour              0
julius_time           0
ta_chi_lag3           0
ta_chi_lag24          0
ta_chi_roll_mean_3    0
ta_chi_roll_std_3     0
dtype: int64

In [105]:
import holidays
# 출퇴근(7-9시, 19-22시) 유무: 1, 0
df4['work_time'] = 0
target_data = [7, 8, 9, 19, 20, 21, 22]
df4.loc[df4['hour'].isin(target_data), 'work_time'] = 1
# 주말 유무(1, 0)
df4['weekend'] = 0
target_data = [5, 6]
df4.loc[df4['dayofweek'].isin(target_data), 'weekend'] = 1
# 공휴일 유무(1, 0)
kr_holidays = holidays.KR(years=range(2023, 2025))
df4['date_only'] = df4['datetime'].dt.date
holiday_dates_set = set(kr_holidays)
df4['holiday'] = df4['date_only'].isin(holiday_dates_set).astype(int)
df4.drop(columns=['date_only'], inplace=True)

In [107]:
df4.isna().sum()

branch_id             0
ta_chi                0
datetime              0
year                  0
month                 0
day                   0
hour                  0
dayofweek             0
dayofyear             0
weekofyear            0
branch_id_encoded     0
sin_hour              0
cos_hour              0
julius_time           0
ta_chi_lag3           0
ta_chi_lag24          0
ta_chi_roll_mean_3    0
ta_chi_roll_std_3     0
work_time             0
weekend               0
holiday               0
dtype: int64

In [109]:
df4.to_csv('./test_heat_add_time_features.csv', index=False, encoding='utf-8')